# 第 17 课｜蒙特卡洛模拟入门

> **数据身份**：固定随机种子生成的模拟教学数据，不是真实证券行情。  
> **环境**：`pkm-quant` · Python 3.11 · 默认离线运行。  
> **课程原始材料**：`17_youtube-WzDKG89SzQg/index.html`  
> **所属模块**：P1-M4 · 随机模型与衍生品风险

## 本节量化问题

随机模拟能回答什么，不能证明什么？

## 准确定义与策略报告位置

蒙特卡洛从明确的概率模型重复抽样，用于估计分布、路径风险和定价期望。

- **核心关系**：蒙特卡洛估计 E[g(X)]≈(1/M)Σg(X_m)，标准误通常按 1/√M 下降。
- **写入报告**：情景分布、尾部概率和模型参数
- **P2 衔接**：P2 第 32、53–54 课：数组、并行和数值性能。


## 运行环境与数据契约

先确认项目、数据目录和数据身份。后续联网数据源必须通过适配层进入，不能改变本 Notebook 的研究计算口径。

In [1]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "data" / "fixtures").exists():
    ROOT = ROOT.parent
DATA_DIR = ROOT / "data" / "fixtures"
OUTPUT_DIR = ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

RANDOM_SEED = 365
print("项目:", ROOT)
print("数据:", DATA_DIR)
print("数据身份: 固定随机种子生成的模拟教学数据，不是真实证券行情")


项目: /Users/mojiawen/Documents/学习资料归档/_暂存/personal_knowledge_manager/量化交易/实验/p1-quant-foundations
数据: /Users/mojiawen/Documents/学习资料归档/_暂存/personal_knowledge_manager/量化交易/实验/p1-quant-foundations/data/fixtures
数据身份: 固定随机种子生成的模拟教学数据，不是真实证券行情


## 模拟终值分布并检查蒙特卡洛误差

参数是模型输入而不是历史真相。增加路径数应使终值均值更稳定，但不会修复错误分布假设。

In [2]:
rng = np.random.default_rng(365)
spot, mu, sigma, years, steps = 100.0, 0.06, 0.20, 1.0, 252
dt = years / steps

def simulate_terminal(path_count):
    shocks = rng.standard_normal((steps, path_count))
    log_steps = (mu - 0.5 * sigma**2) * dt + sigma * np.sqrt(dt) * shocks
    return spot * np.exp(log_steps.sum(axis=0))

terminal_1k = simulate_terminal(1_000)
terminal_10k = simulate_terminal(10_000)
display(pd.DataFrame({
    "paths": [1_000, 10_000],
    "mean_terminal": [terminal_1k.mean(), terminal_10k.mean()],
    "p05": [np.quantile(terminal_1k, 0.05), np.quantile(terminal_10k, 0.05)],
    "p95": [np.quantile(terminal_1k, 0.95), np.quantile(terminal_10k, 0.95)],
}))

,paths,mean_terminal,p05,p95
0,1000,105.851229,74.846449,144.476187
1,10000,106.487701,75.397781,144.771349


## 方法边界与阶段验收

模拟结果只对输入模型和参数成立；增加路径数减少抽样误差，但不减少模型误差。

完成本课后应能：

1. 不看课程材料准确解释核心概念和公式符号；
2. 用小样本手工核对 Notebook 的关键结果；
3. 说明结果能支持什么判断、不能支持什么判断；
4. 指出它在策略报告中的位置，以及仍缺少的数据、成本或稳健性证据。
